In [ ]:
# Setup. Everything below runs the stage-06b CLI itself, so the notebook and the PBS
# job execute identical code.
import logging
import os
import sys

for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import pandas as pd  # noqa: E402
from IPython.display import Image, Markdown, display  # noqa: E402

from scripts.run_corrca_snr_grid import _build_arg_parser, run  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402

logging.basicConfig(
    level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True
)
print("Setup complete.")

# Stage 06b — CorrCA condition comparison (Placebo-fitted filters)

Does a learned CorrCA filter read the 40 Hz ASSR better than averaging the fronto-central
electrodes, and does it separate Placebo from Psilocybin better? The stage-06 question and
the stage-06 grid, asked of the CorrCA filters.

The **subject**-axis join with the CorrCA filters fitted on the **Placebo recordings alone**, every recording of both conditions then projected through them. The contrast (Placebo − Psilocybin) therefore reads Placebo through filters fitted on it and Psilocybin through filters that never saw it, so it leans towards Placebo; the `held_out` rows of the robustness check, each participant read through a refit that never saw them, are the unbiased read. The grid reads the stored filter directly and applies no per-recording sign
flip: the one sign per component was fixed at fit time, positive over the ASSR electrodes.

**The method is documented in
[`docs/analyses/06b-corrca-condition-comparison/`](../../docs/analyses/06b-corrca-condition-comparison/README.md)**
— what CorrCA changes about the stage-06 grid. This notebook does not restate it.

**What this notebook is for.** Checking the workflow cheaply before committing a node
to it. It calls `run()` from
[`scripts/run_corrca_snr_grid.py`](../../scripts/run_corrca_snr_grid.py) — the same function
the PBS job calls — on the small decomposition the fit notebook stored, then shows what it
wrote. **Main workflow only:** one grid cell (the 40 Hz bin, 0–0.5 s), the stored sources
(`full`) and the full-montage store. The job's other cells, signal variants and stores go
through the same code and figures. The numbers mean nothing at 6 participants and 12 s;
the run checks that the tables and figures come out as expected.

See [`corrca_component_analysis_joined_separate.ipynb`](corrca_component_analysis_joined_separate.ipynb) for the same join fitted on both conditions.

**Prerequisite:** run [`corrca_channel_placebo_fit.ipynb`](corrca_channel_placebo_fit.ipynb) first. It stores its fit, and
the leave-subset-out refits the robustness rows read, under `out/placebo_fit/store/`.

## Configuration

In [ ]:
# The decomposition to read: the one the fit notebook stored. N_PCA and N_TIMES must
# match it -- the grid z-scores the same samples the fit did.
EXPERIMENT = "assr"
VARIANT = "corrca_channel_placebo_fit"
N_PCA = 5
BAND = None  # "30-50hz" reads a gamma-band store (BAND_RANGE = (30, 50) in the fit)
STORE_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "06b-corrca-condition-comparison"
    / "out"
    / "placebo_fit"
    / "store"
)
N_TIMES = 3000  # samples (~12 s at 250 Hz)

# One grid cell and the stored sources only: the job's other half-widths, windows and
# signal variants go through the same code and figures.
HALFWIDTHS = [0.0]  # 0 = the single nearest bin; 5.0 = mean over 35-45 Hz
STIMULUS_INTERVALS = ["0:0.5"]  # seconds from onset
SIGNAL_VARIANTS = ["full"]

# Outputs are kept beside the notebook so a run never lands on top of the job's results.
OUT_DIR = (
    ProjectPaths.NOTEBOOKS_DIR / "06b-corrca-condition-comparison" / "out" / VARIANT
)
PLOTS_DIR = OUT_DIR / "plots"
RESULTS_DIR = OUT_DIR / "results"

## Run

`run()` loads the store, reads the wavelet data at the 40 Hz bins for the first `N_TIMES`
samples (memory-mapped, so only that slice is read), runs the three test families and the
robustness of the contrast to the participants, and writes the CSVs and figures. It is the
same function the PBS job calls.

In the log, *the full fit read from the cache vs from the store* should show a difference
of about 1e-3 or less: here the fit and the grid z-score the same samples, so a large value
is a bug, not the small scale.

In [ ]:
args = _build_arg_parser().parse_args(
    [
        "--experiment",
        EXPERIMENT,
        "--variant",
        VARIANT,
        "--n_pca",
        str(N_PCA),
        *(["--band", BAND] if BAND else []),
        "--store_dir",
        str(STORE_DIR),
        "--n_times",
        str(N_TIMES),
        "--lenient_mask",  # the fit's 32-channel subset holds few ASSR electrodes
        # The main workflow only: the full-montage store, raw power.
        "--skip_log_power",
        "--skip_roi",
        "--skip_non_roi",
        "--halfwidths",
        *[str(h) for h in HALFWIDTHS],
        "--stimulus_intervals",
        *STIMULUS_INTERVALS,
        "--signal_variants",
        *SIGNAL_VARIANTS,
        "--save_dir",
        str(PLOTS_DIR),
        "--results_dir",
        str(RESULTS_DIR),
    ]
)
run(args)

## Results

One row per (variant, frequency selection, stimulus window, reference, test family,
source). Nothing is corrected for multiplicity — read `effect` and `same sign` alongside
`p`.

In [ ]:
csv_path = next(RESULTS_DIR.rglob("assr_snr_grid__*.csv"))
results = pd.read_csv(csv_path)
print(f"{csv_path}  —  {len(results)} rows")

display(results.groupby(["family", "variant"]).size().rename("tests").to_frame())
display(
    results.sort_values("p")
    .head(15)
    .loc[
        :,
        [
            c
            for c in (
                "family",
                "variant",
                "source",
                "median",
                "effect",
                "same sign",
                "p",
            )
            if c in results
        ],
    ]
)

## Figures

Written by the shared plotters in
[`src/visualization/assr_snr_plots.py`](../../src/visualization/assr_snr_plots.py), which
the stage-06 grid and the CLI use too — so what is shown here is what the job produces.
`robustness__full.png` is the contrast's robustness to the participants, for CC 1-2: the
p-value of every leave-subset-out rerun, the flip shares per subset size and the refit
topographies against the full fit's.

In [ ]:
for png in sorted(PLOTS_DIR.rglob("*.png")):
    display(Markdown(f"**{png.relative_to(PLOTS_DIR)}**"))
    display(Image(filename=str(png)))